In [ ]:
import os
import glob
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from safetensors.torch import load_file
import numpy as np
import pandas as pd
from pathlib import Path

# Set style
sns.set_theme(style="whitegrid")

# ============================================================================
# CONFIGURATION - Edit these parameters
# ============================================================================

# Dataset configuration (use HuggingFace name)
DATASET_NAME = "l11p/VQAv2-COCO-restval-ICQ"  # or any HuggingFace dataset name
# Auto-detect prefix from dataset name
DATASET_PREFIX = DATASET_NAME.split("/")[-1].replace(".", "_").replace("-", "_").lower()

# Dataset splits to analyze (automatically adds prefix)
DATASET_VARIANTS = ["img", "txt"]  # Will become {prefix}_img, {prefix}_txt

# Models to analyze (use HuggingFace names)
MODELS = {
    "Pretrained": "l11p/llava-v1.5-7b-pretrained-liuhaotian",
    "Finetuned": "llava-hf/llava-1.5-7b-hf"
}

# Pooling methods to analyze
POOLING_METHODS = ["last", "mean_image"]

# ============================================================================
# PATHS - Aligned with Python scripts
# ============================================================================

RESULTS_DIR = Path("../results")
ID_RESULTS_DIR = RESULTS_DIR / "model_comparison" / "intrinsic_dimension"

print(f"📊 Configuration:")
print(f"  Dataset: {DATASET_NAME}")
print(f"  Prefix: {DATASET_PREFIX}")
print(f"  Variants: {[f'{DATASET_PREFIX}_{v}' for v in DATASET_VARIANTS]}")
print(f"  Models: {list(MODELS.keys())}")
print(f"  Pooling: {POOLING_METHODS}")
print(f"  ID Results: {ID_RESULTS_DIR}")

In [ ]:
def load_id_data(dataset_variant, pooling_method, models_dict):
    """
    Load intrinsic dimension data for all models.
    
    Args:
        dataset_variant: 'img' or 'txt'
        pooling_method: 'last', 'mean_image', etc.
        models_dict: Dictionary of {label: huggingface_name}
    
    Returns:
        Dictionary of {model_label: id_scores_array}
    """
    data = {}
    dataset_name = f"{DATASET_PREFIX}_{dataset_variant}"
    
    for label, hf_name in models_dict.items():
        model_id = hf_name.split("/")[-1]
        
        # Path format: {model_id}_{dataset}_{pooling}_intrinsic_dimension.safetensors
        filename = f"{model_id}_{dataset_name}_{pooling_method}_intrinsic_dimension.safetensors"
        filepath = ID_RESULTS_DIR / filename
        
        if not os.path.exists(filepath):
            print(f"⚠️  Warning: File not found: {filepath}")
            print(f"    Hint: Run intrinsic_dimension.py with this dataset and model first")
            continue
            
        try:
            loaded = load_file(str(filepath))
            if "intrinsic_dimension" in loaded:
                id_scores = loaded["intrinsic_dimension"].float().numpy()
                data[label] = id_scores
                print(f"✅ Loaded {label}: {filename}")
            else:
                print(f"⚠️  Warning: 'intrinsic_dimension' key not found in {filename}")
        except Exception as e:
            print(f"⚠️  Error loading {filepath}: {e}")
            
    return data

def plot_id_evolution(data, dataset_variant, pooling_method):
    """Plot intrinsic dimension evolution across layers for all models."""
    if not data:
        print(f"⚠️  No data available for {dataset_variant} with {pooling_method} pooling")
        return
        
    plt.figure(figsize=(14, 7))
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(data)))
    
    for (model_label, scores), color in zip(data.items(), colors):
        layers = np.arange(len(scores))
        plt.plot(layers, scores, marker='o', label=model_label, linewidth=2.5, 
                alpha=0.8, color=color, markersize=5)
        
    plt.title(f"Intrinsic Dimension Evolution: {dataset_variant.upper()} ({pooling_method})", 
             fontsize=18, fontweight='bold', pad=15)
    plt.xlabel("Layer Index", fontsize=14, fontweight='medium')
    plt.ylabel("Intrinsic Dimension", fontsize=14, fontweight='medium')
    plt.grid(True, linestyle='--', alpha=0.5, linewidth=0.8)
    plt.legend(fontsize=12, framealpha=0.9, shadow=True)
    plt.tight_layout()
    
    # Save plot
    plots_dir = Path(f"../plots/intrinsic_dimension_{DATASET_PREFIX}")
    plots_dir.mkdir(parents=True, exist_ok=True)
    save_path = plots_dir / f"id_evolution_{dataset_variant}_{pooling_method}.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor='white')
    print(f"✅ Saved plot to {save_path}")
    
    plt.show()

def plot_id_comparison_grid(all_data):
    """Plot all combinations in a grid for easy comparison."""
    n_variants = len(DATASET_VARIANTS)
    n_pooling = len(POOLING_METHODS)
    
    fig, axes = plt.subplots(n_variants, n_pooling, figsize=(7*n_pooling, 6*n_variants))
    if n_variants == 1:
        axes = axes.reshape(1, -1)
    if n_pooling == 1:
        axes = axes.reshape(-1, 1)
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(MODELS)))
    
    for i, variant in enumerate(DATASET_VARIANTS):
        for j, pooling in enumerate(POOLING_METHODS):
            ax = axes[i, j]
            key = (variant, pooling)
            data = all_data.get(key, {})
            
            if data:
                for (model_label, scores), color in zip(data.items(), colors):
                    layers = np.arange(len(scores))
                    ax.plot(layers, scores, marker='o', label=model_label, 
                           linewidth=2, alpha=0.8, color=color, markersize=4)
                
                ax.set_title(f"{variant.upper()} - {pooling}", fontsize=14, fontweight='bold')
                ax.set_xlabel("Layer Index", fontsize=11)
                ax.set_ylabel("Intrinsic Dimension", fontsize=11)
                ax.grid(True, linestyle='--', alpha=0.5)
                ax.legend(fontsize=10)
            else:
                ax.text(0.5, 0.5, f"No data\n{variant} - {pooling}", 
                       ha='center', va='center', transform=ax.transAxes, 
                       fontsize=12, color='gray')
                ax.axis('off')
    
    plt.suptitle(f"Intrinsic Dimension: {DATASET_NAME}", 
                fontsize=18, fontweight='bold', y=0.995)
    plt.tight_layout()
    
    # Save grid plot
    plots_dir = Path(f"../plots/intrinsic_dimension_{DATASET_PREFIX}")
    plots_dir.mkdir(parents=True, exist_ok=True)
    save_path = plots_dir / f"id_comparison_grid.png"
    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor='white')
    print(f"✅ Saved grid plot to {save_path}")
    
    plt.show()

In [ ]:
# ============================================================================
# GENERATE PLOTS
# ============================================================================

print("="*80)
print("LOADING INTRINSIC DIMENSION DATA")
print("="*80)

# Load all data combinations
all_data = {}
for variant in DATASET_VARIANTS:
    for pooling in POOLING_METHODS:
        print(f"\n📊 Loading: {variant} + {pooling}")
        data = load_id_data(variant, pooling, MODELS)
        if data:
            all_data[(variant, pooling)] = data

print("\n" + "="*80)
print("GENERATING INDIVIDUAL PLOTS")
print("="*80)

# Plot individual combinations
for variant in DATASET_VARIANTS:
    for pooling in POOLING_METHODS:
        key = (variant, pooling)
        if key in all_data:
            print(f"\n📈 Plotting: {variant} + {pooling}")
            plot_id_evolution(all_data[key], variant, pooling)

print("\n" + "="*80)
print("GENERATING COMPARISON GRID")
print("="*80)

# Plot comparison grid
if all_data:
    plot_id_comparison_grid(all_data)
else:
    print("⚠️  No data found! Please run intrinsic_dimension.py first.")
    print(f"   Example command:")
    print(f"   python scripts/representations_interpretability/intrinsic_dimension.py \\")
    print(f"     --results-dir results \\")
    print(f"     --representations-dir $HOME/scratch \\")
    print(f"     --dataset-path-or_name {DATASET_NAME} \\")
    print(f"     --models {list(MODELS.values())[0]} \\")
    print(f"     --pooling-methods {' '.join(POOLING_METHODS)} \\")
    print(f"     --save-representations")

print("\n" + "="*80)
print("✅ DONE!")
print("="*80)